# Decorators
![decorators](Images/decorators.png)

## Use cases

There can be use cases when the same actions should be repeated when calling different functions
   * checking arguments
   * counting function calls
   * requiring authentication (e.g., web-servers)
   * timing
   * logging
   * ...

In [65]:
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

In [66]:
factorial(5)

120

## Example 1: Counting function calls

In [67]:
n_calls = 0
n_calls_rec = 0

def factorial(n):
    global n_calls
    n_calls += 1
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

def factorial_rec(n):
    global n_calls_rec
    n_calls_rec += 1
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial_rec(n-1)

In [68]:

factorial_rec(150)

print(f"factorial called: {n_calls} with res {factorial(150)}")
print(f"factorial called: {n_calls_rec} with res {factorial_rec(150)}")
print(factorial(150) == factorial_rec(150))

factorial called: 0 with res 57133839564458545904789328652610540031895535786011264182548375833179829124845398393126574488675311145377107878746854204162666250198684504466355949195922066574942592095735778929325357290444962472405416790722118445437122269675520000000000000000000000000000000000000
factorial called: 150 with res 57133839564458545904789328652610540031895535786011264182548375833179829124845398393126574488675311145377107878746854204162666250198684504466355949195922066574942592095735778929325357290444962472405416790722118445437122269675520000000000000000000000000000000000000
True


### Not good:
   * Function has to be modified
   * Global variable is used (actually it might have been avoided...)

## Example 2: Warning for slow execution

In [69]:
import time

def factorial(n, timelimit=1e-7):
    t0 = time.time()    
    fact = 1
    for i in range(1,n+1):
        fact *= i
    dt = time.time() - t0
    if dt > timelimit:
        print(f'Warning! factorial took {dt} seconds')
    else:
        print(f'Info: factorial took {dt} seconds')
    return fact

In [70]:
factorial(4)
factorial(5)

Warning! factorial took 2.1457672119140625e-06 seconds
Info: factorial took 0.0 seconds


120

### Not good, again:
   * *Each* function has to be modified
   * Activating and deactivating timing check is not immediate

## A better idea?
let us first recollect useful Python features about functions

### 1. Assigning functions to variables
   

In [71]:
def plus_one(number):
    return number + 1

add_one = plus_one
add_one(5)

6

### 2. Defining Functions Inside other Functions

In [72]:
def plus_one(number):

    def add_one(number):
        return number + 1

    result = add_one(number)
    return result

plus_one(4)

5

### 3. Passing Functions as Arguments to other Functions

In [73]:
def plus_one(number):
    return number + 1

def function_call(function):
    number_to_add = 5
    return function(number_to_add)

function_call(plus_one)

6

### 4. Functions Returning other Functions

In [74]:
def hello_function():
    
    def say_hi():
        return "Hi"
    
    return say_hi

hello = hello_function()
hello()

'Hi'

### 5. Nested Functions have access to the Enclosing Function's Variable Scope

In [75]:
def print_message(message):
    
    def message_sender():
        print(message)

    message_sender()

print_message("A message")

A message


## A better idea!

* Create a function which "decorates" the original function adding the needed actions

* "decorate" = make (something) that looks more attractive by adding extra items or images to it

In [76]:
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

factorial(5)

120

In [77]:
import time

def warning_slow_decorator(function_to_decorate):
    
    def warning_slow_decorated(n):
        timelimit = 0.001
        t0 = time.time()    
        result = function_to_decorate(n)
        dt = time.time() - t0

        if dt > timelimit:
            print(f'Warning! factorial took {dt} seconds')
        else:
            print(f'Info: factorial took {dt} seconds')
        return result
    
    return warning_slow_decorated

decorated_factorial = warning_slow_decorator(factorial)

In [78]:
decorated_factorial(5)

Info: factorial took 9.5367431640625e-07 seconds


120

## Good!
* Each function can be decorated by applying the decorator function
* The same decorator works for each function
* Ok, not proprely EACH function, but now...

In [79]:
def warning_slow_decorator(function_to_decorate):

    def warning_slow_decorated(*args, **kwargs):
        print(args, kwargs)
        timelimit = 0.001
        t0 = time.time()    
        result = function_to_decorate(*args, **kwargs)
        dt = time.time() - t0

        if dt > timelimit:
            print(f'Warning! factorial took {dt} seconds')
        else:
            print(f'Info: factorial took {dt} seconds')
        return result

    return warning_slow_decorated

    return fact


decorated_factorial = warning_slow_decorator(factorial)

In [80]:
decorated_factorial(5)

(5,) {}
Info: factorial took 1.1920928955078125e-06 seconds


120

## Can we do better? 

* We hope ... otherwise we have to change all function names
* Yes, Python decorators make the decorating syntax very readable!

In [81]:
@warning_slow_decorator
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i  
    return fact

factorial(5)

(5,) {}
Info: factorial took 2.1457672119140625e-06 seconds


120

* You can easily use decorators written for your favourite library/framework/... 
* Probably you will not need decorators when you are beginner
* But you will like them much when you become more skilled

## What about counting function calls?

The case is a bit more difficult if we want to avoid the global variable

In [82]:
def counting_call_decorator(f):
    n_calls = 0
    def counting_call_decorated(*args, **kwargs):
        nonlocal n_calls
        n_calls += 1
        result = f(*args, **kwargs)
        return result, n_calls
    return counting_call_decorated

@counting_call_decorator
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i  
    return fact

print(factorial(3))
print(factorial(4))
print(factorial(5))

(6, 1)
(24, 2)
(120, 3)


## Note
* The result of function is modified adding n_calls
* But, why does it work? i.e. why does the local variable 
(for the outer function) n_calls survives after the function call?

### This is a consequence of the so-called *Closure*

In a nutshell, when defining the decorator
* n_calls is visible in the inner function (see point 5 above)
* and since the outer function returns the inner function this variable must continue to be accessible 
* nonlocal is needed to give since n_calls is immutable (otherwise using mutable types is a simpler option)
* *Closures* are difficult to grasp at the first time, don't worry!

## Avoiding closures

* Is it possible? Of course... assign attribute to the function itself is on option!

In [83]:
def counting_call_decorator(f):
    def counting_call_decorated(*args, **kwargs):
        counting_call_decorated.n_calls += 1
        result = f(*args, **kwargs)
        return result
    counting_call_decorated.n_calls = 0
    return counting_call_decorated

@counting_call_decorator
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

print(factorial(3), factorial.n_calls)
print(factorial(4), factorial.n_calls)
print(factorial(5), factorial.n_calls)


6 1
24 2
120 3


## Decorators with arguments

What if we want to have a parametric decorator, i.e. pass arguments to decorator itself

In [84]:
import time
def warning_slow_decorator_withargs(timelimit):
    def warning_slow_decorator(function_to_decorate):
        def warning_slow_decorated(*args, **kwargs):
            t0 = time.time()    
            result = function_to_decorate(*args, **kwargs)
            dt = time.time() - t0
            if dt > timelimit:
                print(f'Warning! factorial took {dt} seconds')
            else:
                print(f'Info: factorial took {dt} seconds')
            return result
        return warning_slow_decorated
    return warning_slow_decorator

@warning_slow_decorator_withargs(1)
def factorial(n):
    fact = 1
    for i in range(1,n+1): fact *= i  
    return fact

print(factorial(4))
print(factorial(3))
print(factorial(5))

Info: factorial took 1.9073486328125e-06 seconds
24
Info: factorial took 9.5367431640625e-07 seconds
6
Info: factorial took 7.152557373046875e-07 seconds
120


## Multiple decoration

* It can be very useful to superimpose the actions produced by decorators
* How does it work? Sandwich inside sandwich...
* Beware: if a decorator modifies arguments before calling the decorated function or the returned variables, it is possible that combining decorators does not work

In [85]:
@counting_call_decorator
@warning_slow_decorator_withargs(1)
def factorial(n):
    fact = 1
    for i in range(1,n+1): fact *= i  
    return fact

print(factorial(4))
print(factorial(3))

Info: factorial took 1.6689300537109375e-06 seconds
24
Info: factorial took 7.152557373046875e-07 seconds
6


## Preserving metadata
* Use functools.wraps
* Try with and without it

In [86]:
import functools

def counting_call_decorator(f):
    n_calls = 0
    #@functools.wraps(f)
    def counting_call_decorated(*args, **kwargs):
        """counting calls decorator"""
        nonlocal n_calls
        n_calls += 1
        result = f(*args, **kwargs)
        return result, n_calls
    return counting_call_decorated

@counting_call_decorator
def factorial(n):
    """returns factorial of number n"""
    fact = 1
    for i in range(1,n+1): fact *= i  
    return fact

print("factorial name :", factorial.__name__)
print("factorial doc  :", factorial.__doc__)

factorial name : counting_call_decorated
factorial doc  : counting calls decorator


## Decorator package
* Wrapper functions, wrapper of wrapper functions, closure, tricky logic... is there any easier way to define decorators?
* Use decorator module

In [87]:
from decorator import decorator

@decorator
def warning_slow_decorator(func, timelimit=60, *args, **kwargs):
    t0 = time.time()
    result = func(*args, **kwargs)
    dt = time.time() - t0
    if dt > timelimit:
        print(f'Warning! factorial took {dt} seconds')
    else:
        print(f'Info: factorial took {dt} seconds')
    return result

@warning_slow_decorator(timelimit=1e-7)
def factorial(n):
    fact = 1
    for i in range(1,n+1): fact *= i  
    return fact

factorial(4)
factorial(5)

Warning! factorial took 1.9073486328125e-06 seconds
Warning! factorial took 9.5367431640625e-07 seconds


120

## Hands-on 12.1
* Write a decorator for functions with 1 arguments to force the argument to be a natural number

In [88]:
def natural_number_decorator(f):
    def natural_number_decorated(n):
        if type(n) is int and n >= 0: # Se è un numero naturale faccio la chiamata a funzione
            return f(n)
        else:
            raise ValueError("Input must be a natural number (non-negative integer).")
            
    return natural_number_decorated

@natural_number_decorator
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

print(f"Fattoriale: {factorial(0)}")

Fattoriale: 1


## Hands-on 12.1 - BIS
* Write the parametric decorator imposing an optional maximum value for the input integer
* First try with the standard decorator definition, then experiment with decorator module

In [90]:
def max_number_decorator(f):
    def max_number_decorated(*args, **kwargs):
        #print(args, kwargs)
        #print(len(args), len(kwargs))
        
        # Se c'è anche il limite -> 2 argomenti ( args o kwargs )
        if (len(args) > 1 and args[0] < args[1]) or (kwargs.get('nMax') is not None and args[0] < kwargs['nMax']): 
            return f(args[0])
        elif len(args) == 1 and kwargs.get('nMax') is None: # Se c'è solamente il numero -> 1 argomento
            return f(args[0])
        else: # Casi di errore
            if(len(args) > 1):
                raise ValueError(f"Input must be less than {args[1]}.")
            else:
                raise ValueError(f"Input must be less than {kwargs['nMax']}.")
            
    return max_number_decorated

@max_number_decorator
def factorial(n):
    fact = 1
    for i in range(1,n+1):
        fact *= i
    return fact

print(f"Fattoriale: {factorial(10)}")

Fattoriale: 3628800
